# MVSS-Net project - Colab workflow

See `docs/COLAB_RUNBOOK.md` for the full explanation, licence notes and caveats.

**Reality check:** Module 1 retraining and Module 2 training have not been run on a GPU in this repository. This notebook only automates the documented commands; it does not contain results.

In [ ]:
import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Module 1 at 512 px will be extremely slow.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO = '/content/drive/MyDrive/mvss/repo'
DATA = '/content/drive/MyDrive/mvss/data'
EXPERIMENTS = '/content/drive/MyDrive/mvss/experiments'
os.makedirs(EXPERIMENTS, exist_ok=True)
%cd $REPO
!pip install -q -r requirements-app.txt

In [ ]:
import hashlib, pathlib
expected = {
    'ckpt/mvssnet_casia.pt': '080BC6C3AAE59F748B547DBF090786FE9D31A6E50749DAAA40871E298D6A7E50',
    'results/manifests/casia2_groupaware_split.csv': 'BE5ACAB205E294A563397C8E6756A6CA429DAD01E3F4FEE5EEBDC84E2593E49C',
}
for name, digest in expected.items():
    actual = hashlib.sha256(pathlib.Path(name).read_bytes()).hexdigest().upper()
    status = 'OK' if actual == digest else 'MISMATCH - stop and investigate'
    print(name, status)
assert all(
    hashlib.sha256(pathlib.Path(n).read_bytes()).hexdigest().upper() == d
    for n, d in expected.items()
), 'Input hash mismatch: do not train'

## Module 1 - smoke check

Four images, one epoch. Confirms the data pipeline and artefact writing on Colab before a long run.

In [ ]:
!python training/train_mvss_groupaware.py \
  --data-root $DATA \
  --manifest results/manifests/casia2_manifest.csv \
  --split-csv results/manifests/casia2_groupaware_split.csv \
  --output-dir $EXPERIMENTS/module1/smoke \
  --device cuda --smoke --batch-size 2

## Module 1 - full training (resumable)

Set `RESUME` to `last.pt` only after a disconnect. Never pass `--overwrite` to a run you want to keep.

In [ ]:
RUN = f'{EXPERIMENTS}/module1/groupaware_v1'
RESUME = ''  # e.g. f'{RUN}/last.pt' after a disconnect
resume_arg = f'--resume {RESUME}' if RESUME else ''
!python training/train_mvss_groupaware.py \
  --data-root $DATA \
  --output-dir $RUN \
  --device cuda --epochs 40 --batch-size 4 --num-workers 2 --image-size 512 $resume_arg

## Module 1 - independent test evaluation

Uses the trained checkpoint, not the preserved original. Thresholds are fixed from validation-only selection.

In [ ]:
!python evaluation/evaluate_groupaware_checkpoint.py \
  --checkpoint $RUN/mvssnet_groupaware.pt \
  --data-root $DATA \
  --split test --device cuda --batch-size 4 \
  --output results/evaluation/module1_groupaware_test_metrics.json \
  --per-image-csv results/evaluation/module1_groupaware_test_per_image.csv

## Module 2 - guarded by dataset availability

Do not download GenImage/Synthbuster without explicit approval (see runbook §4). This cell exits if the folders are absent.

In [ ]:
from pathlib import Path
real_dir = Path(DATA) / 'synthbuster' / 'real'
ai_dir = Path(DATA) / 'synthbuster' / 'ai'
if not real_dir.is_dir() or not ai_dir.is_dir():
    print('Module 2 dataset not present:', real_dir, ai_dir)
    print('No training has been run. Obtain approval and a licence before downloading.')
else:
    RUN2 = f'{EXPERIMENTS}/module2/synthbuster_v1'
    !python training/train_ai_detector.py \
      --real-dir $real_dir --ai-dir $ai_dir --output-dir $RUN2 \
      --holdout-generators "Midjourney_v5,DALL-E_3" \
      --backbone efficientnet_b0 --pretrained \
      --input-size 224 --epochs 15 --batch-size 32 \
      --device cuda --dataset-name Synthbuster \
      --dataset-license "CC BY-NC-SA 4.0" \
      --provenance "Zenodo 10.5281/zenodo.10066460; real images RAISE-1k"

## Module 3 - synthetic degradation stress test (CPU)

Results are synthetic. They are not evidence about real captured conversations.

In [ ]:
!python evaluation/evaluate_chat_degradation.py --dataset $DATA/chat_synthetic_v1

## Before reporting

- Copy back only JSON/CSV/log artefacts, never checkpoints or datasets.
- Label synthetic Module 3 results as synthetic.
- Report held-out-generator Module 2 metrics separately from in-distribution metrics.
- Do not compare the group-aware test split with the historical CASIA test split as if they were the same benchmark.